# FireFusion — ENSO Processor Documentation

**Project:** FireFusion  
**Stream:** Data Engineering  
**Task:** ENSO Processor Documentation  
**Contributor:** Pitambri Sareen  
**Pipeline Owner:** FireFusion Data Engineering Stream  
**Pipeline:** El Niño–Southern Oscillation (ENSO) NOAA CPC ONI Data Pipeline  
**Pipeline Script:** `fetch_enso.py`

---

## Document Purpose

This document describes the **NOAA CPC Oceanic Niño Index (ONI) ENSO data pipeline** implemented for the FireFusion Data Engineering stream.

The pipeline extracts monthly Sea Surface Temperature (SST) anomaly index information from the **NOAA Climate Prediction Center (CPC)** public endpoint and converts the source data into structured CSV outputs suitable for downstream FireFusion processing.

The resulting ENSO information is intended to provide macro-climate features, including:

- `oni_anomaly`
- `oni_lag6m`
- `enso_phase`

These features can be used to pre-condition fuel dryness and support long-term bushfire risk prediction for Victoria.

> **Scope:** This document focuses on the implemented ENSO/ONI pipeline, its data flow, source and output schema, processing logic, edge cases, validation, execution procedure and automation.

## 1. Pipeline Overview

The ENSO processor follows a source-to-output data pipeline:

```text
NOAA CPC ONI Endpoint
        |
        | HTTP GET
        v
+------------------------------+
| fetch_enso.py                |
|                              |
| 1. Extract raw ONI data      |
| 2. Preserve source data      |
| 3. Align dates               |
| 4. Generate time_id          |
| 5. Engineer ENSO features    |
| 6. Assign enso_id            |
+------------------------------+
        |
        +--------------------+
        |                    |
        v                    v
   data/raw/           data/processed/
        |                    |
        |                    v
        |             Processed CSV
        |                    |
        +----------+---------+
                   |
                   v
             datasets/enso/
                   |
                   v
        FireFusion downstream use
```

The pipeline is designed so that the original downloaded source can be retained for audit lineage while processed data is delivered in CSV format for downstream use.

## 2. Pipeline Name

**El Niño–Southern Oscillation (ENSO) NOAA CPC ONI Data Pipeline**

### Pipeline Script

```text
fetch_enso.py
```

The script is responsible for fetching the NOAA CPC ONI data and producing the raw and processed outputs described in this document.

## 3. Data Source

The pipeline uses the following external source:

| Attribute | Details |
|---|---|
| Source | NOAA Climate Prediction Center (CPC) |
| Provider | NOAA Physical Sciences Laboratory |
| Dataset | Oceanic Niño Index (ONI) |
| API / Dataset URL | `https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt` |
| Collection method | HTTP GET |
| Python retrieval library | `urllib.request` |
| Source format | Space-delimited ASCII text |
| Output format | CSV |
| Refresh frequency | Monthly |
| Pipeline owner | FireFusion Data Engineering stream |

The source is accessed programmatically rather than requiring a developer to manually download the data.

## 4. Input Data

The processor retrieves data directly from:

```text
https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt
```

The incoming source is a space-delimited ASCII dataset.

The raw response is preserved unchanged for audit and data-lineage purposes using the following naming convention:

```text
data/raw/noaa_cpc_enso_oni_raw_<YYYYMMDD>.ascii
```

Preserving the raw source provides a reference point between the external NOAA data and the processed FireFusion dataset.

## 5. Processing Workflow

The pipeline performs the following major processing stages:

### 5.1 Extraction

The processor performs an HTTP GET request to the NOAA CPC endpoint and retrieves the raw ONI data.

The downloaded source is stored in `data/raw/` using a date-stamped filename.

### 5.2 Date Alignment

The source `SEAS` values are converted into standard month information.

The processing workflow handles seasonal abbreviations such as:

```text
DJF
NDJ
...
```

and constructs the standard month-start timestamp:

```text
YYYY-MM-01 00:00:00
```

This creates a consistent temporal representation for downstream processing.

### 5.3 Master Time Key

The pipeline generates the `time_id` field according to the FireFusion master calendar architecture.

The resulting key follows the documented `YYYYMMDDHH` representation and aligns ENSO observations with the central `Time_Registry`.

### 5.4 Feature Engineering

The processor derives:

- `oni_lag6m` — the six-month prior ONI anomaly;
- `enso_phase` — the categorical ENSO state.

The documented classification rule includes:

```text
ONI >= +0.5°C → El Nino
```

The exact rules used for the remaining phase categories should remain aligned with the current implementation.

### 5.5 Primary Key Assignment

A sequential `enso_id` is assigned:

```text
1, 2, 3, ...
```

This is used as the primary identifier for ENSO records.

### 5.6 Output Delivery

The processed data is written to both the processed-data location and the ENSO dataset location.

## 6. Feature Engineering

### `oni_anomaly`

Represents the Oceanic Niño Index anomaly and captures the Sea Surface Temperature deviation represented by the ONI dataset.

The documented expected range is approximately:

```text
-3.0°C to +3.0°C
```

### `oni_lag6m`

Represents the ONI anomaly from six months earlier.

This feature is intended to provide a lagged climate signal that can support fuel-drying pre-conditioning.

The first six records may not have a six-month historical value and are therefore allowed to be null.

### `enso_phase`

Represents the derived categorical ENSO state.

The documented categories are:

```text
El Nino
La Nina
Neutral
```

The documented El Niño condition is:

```text
ONI >= +0.5°C
```

> **Implementation note:** This documentation records the classification information provided for the current pipeline. Any additional thresholds or rules for `La Nina` and `Neutral` should be kept consistent with the implementation of `fetch_enso.py`.

## 7. Output Schema

The processed ENSO dataset uses the following schema:

| Column | Description | Type | Unit / Range | Null Allowed | Notes |
|---|---|---|---|---|---|
| `enso_id` | Primary key | INTEGER | 1 to N | No | Sequential entity identifier |
| `time_id` | Universal master calendar key | INTEGER | `YYYYMMDDHH` | No | Foreign key aligned with `Time_Registry` |
| `datetime_record` | Standard month-start timestamp | TIMESTAMP | `YYYY-MM-01 00:00:00` | No | Month-start alignment |
| `record_year_month` | Year-month string | VARCHAR | `YYYY-MM` | No | Example: `2019-12` |
| `oni_anomaly` | Oceanic Niño Index anomaly | NUMERIC | `-3.0` to `+3.0°C` | No | Sea Surface Temperature deviation |
| `enso_phase` | Active ENSO phase | VARCHAR | `El Nino`, `La Nina`, `Neutral` | No | Derived categorical state |
| `oni_lag6m` | Six-month prior ONI anomaly | NUMERIC | `-3.0` to `+3.0°C` | Yes | First six rows may be null |
| `original_source` | Data-lineage origin | VARCHAR | `NOAA_CPC_ONI` | No | Standard lineage tracking |

## 8. Output Locations

The pipeline produces the following files:

### Raw Storage

```text
data-engineering/data/raw/noaa_cpc_enso_oni_raw_<YYYYMMDD>.ascii
```

The raw source is preserved unchanged.

### Processed Target

```text
data-engineering/data/processed/noaa_cpc_enso_oni_processed_<YYYYMMDD>.csv
```

This is the processed CSV output.

### Dataset Target

```text
data-engineering/datasets/enso/noaa_cpc_enso_oni_<YYYYMMDD>.csv
```

This provides the ENSO dataset in the project dataset area for downstream consumption.

## 9. Data Lineage

The pipeline maintains a clear source-to-output lineage:

```text
NOAA CPC
   |
   | HTTP GET
   v
Raw ASCII file
   |
   | Parse / align / transform
   v
Processed ENSO CSV
   |
   | Dataset delivery
   v
datasets/enso/
   |
   v
FireFusion downstream components
```

The `original_source` field records:

```text
NOAA_CPC_ONI
```

This allows processed records to retain their source identity within the FireFusion data architecture.

## 10. Temporal Alignment

Temporal consistency is important because ENSO data may later be joined with other FireFusion environmental datasets.

The pipeline standardises records around a month-start timestamp:

```text
YYYY-MM-01 00:00:00
```

It also generates:

```text
record_year_month
```

using the:

```text
YYYY-MM
```

format.

The `time_id` provides the corresponding master calendar key and is intended to link the ENSO data with the project's `Time_Registry` architecture.

## 11. Edge Cases and Data-Quality Considerations

| Edge Case | Potential Impact | Handling / Consideration |
|---|---|---|
| NOAA endpoint unavailable | No new source data | Pipeline should report retrieval failure |
| Network timeout | Incomplete retrieval | Request should fail rather than produce misleading output |
| Empty response | Empty processed dataset | Validate before output delivery |
| Source format changes | Parsing failure | Inspect source and update parser |
| Missing ONI values | Incorrect derived features | Validate before feature engineering |
| First six `oni_lag6m` records | No six-month history | Null values are expected |
| Invalid seasonal code | Incorrect date alignment | Validate `SEAS` values |
| Duplicate observations | Incorrect time-series features | Detect during validation |
| Unexpected columns | Pipeline compatibility issue | Validate source schema |
| Invalid ONI values | Incorrect phase/features | Apply range/data-quality validation |

## 12. Validation

The pipeline should be validated at several points.

### Source Validation

Confirm that:

- the NOAA endpoint responds successfully;
- the downloaded file is not empty;
- the expected source structure is present.

### Transformation Validation

Confirm that:

- seasonal/date values are converted correctly;
- `datetime_record` follows month-start alignment;
- `record_year_month` uses `YYYY-MM`;
- `time_id` follows the master calendar convention;
- `oni_lag6m` is correctly shifted by six months.

### Output Validation

Confirm that:

- `enso_id` is sequential;
- required fields are populated;
- the first six `oni_lag6m` records are handled as expected;
- `enso_phase` contains valid categories;
- `original_source` is populated as `NOAA_CPC_ONI`;
- raw and processed files exist in their expected locations.

### Evidence Placeholder

> **Insert screenshot:** Successful execution of `fetch_enso.py` showing the generated raw and processed files.

> **Insert screenshot:** Processed CSV opened/inspected to demonstrate the expected schema and derived fields.

## 13. Local Execution Runbook

From the appropriate project directory, run:

```bash
python fetch_enso.py
```

The processor should retrieve the NOAA CPC ONI source and generate the configured raw, processed and dataset outputs.

### Pre-run Checks

Before execution:

1. Confirm Python is available.
2. Confirm the project directory is correct.
3. Confirm required dependencies are installed.
4. Confirm the NOAA endpoint is accessible.
5. Confirm the expected output directories exist or are created by the script.

### Post-run Checks

After execution:

```bash
ls -lah data/raw/
ls -lah data/processed/
ls -lah datasets/enso/
```

Confirm that date-stamped raw and processed files have been generated.

## 14. Dependencies

The automated workflow currently installs:

```bash
python -m pip install --upgrade pip
pip install pandas
```

The pipeline uses Python and the documented `urllib.request` mechanism for HTTP retrieval.

The exact dependency list should remain aligned with the current repository implementation and should be updated if additional libraries are introduced.

## 15. GitHub Actions Automation

The ENSO pipeline has been configured for automated execution using GitHub Actions.

The current workflow is designed with:

```text
Workflow: ENSO Data Pipeline
Trigger:
  - workflow_dispatch
  - scheduled execution

Schedule:
  - 0 0 * * *
```

The workflow uses:

```text
ubuntu-latest
Python 3.12
```

and installs pandas before executing:

```bash
python fetch_enso.py
```

After execution, the workflow checks the generated directories:

```bash
ls -lah data/raw/
ls -lah data/processed/
ls -lah datasets/enso/
```

This provides a basic automated verification that the expected files have been generated.

## 16. GitHub Actions Workflow

The current workflow follows this sequence:

```text
Scheduled / Manual Trigger
          |
          v
Checkout Repository
          |
          v
Set up Python 3.12
          |
          v
Install pandas
          |
          v
Run fetch_enso.py
          |
          v
Check data/raw/
          |
          v
Check data/processed/
          |
          v
Check datasets/enso/
```

### Workflow Configuration

```yaml
name: ENSO Data Pipeline

on:
  workflow_dispatch:

  schedule:
    - cron: "0 0 * * *"

permissions:
  contents: read

jobs:
  run-enso-pipeline:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.12"

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install pandas

      - name: Run ENSO pipeline
        run: |
          python fetch_enso.py

      - name: Check generated files
        run: |
          echo "Raw files:"
          ls -lah data/raw/

          echo "Processed files:"
          ls -lah data/processed/

          echo "ENSO datasets:"
          ls -lah datasets/enso/
```

> **Automation note:** The workflow is currently configured with the schedule shown above. The project team should verify that the scheduled UTC execution time matches the intended operational refresh time.

## 17. Automation Evidence

Recommended PR evidence:

### Evidence 1 — Successful Workflow

Insert a screenshot of the GitHub Actions run showing the workflow completed successfully.

### Evidence 2 — Pipeline Execution

Show the `Run ENSO pipeline` step completing without errors.

### Evidence 3 — Generated Files

Show the `Check generated files` step listing:

```text
data/raw/
data/processed/
datasets/enso/
```

This provides evidence that the automation is not only triggering the script but also producing the expected file outputs.

## 18. Development Contribution

The ENSO processor work involved investigation, implementation, validation and automation rather than documentation alone.

### Source Investigation

The NOAA CPC ONI endpoint was investigated and selected as the external source for the ENSO climate signal.

### Raw Data Investigation

The available ENSO-related datasets were inspected to understand their structure and identify data-quality issues before processing.

### Data Cleaning

The cleaning workflow was developed and tested across:

- `nino34.csv`
- `oni.csv`
- `soi.csv`

The observed raw `nino34.csv` data included the sentinel value `-99.99`, which was identified during inspection and considered during the cleaning process.

### Exploratory Analysis

The `o1_elnino_exploration.ipynb` notebook was used to investigate the ENSO data and support decisions about the processing workflow.

### Pipeline Development

The work progressed from exploratory investigation to a repeatable scripted workflow using separate processing stages.

### Logging

Logging was incorporated into the processing workflow to improve visibility of execution and make failures easier to identify.

### Automation

Work was undertaken to configure GitHub Actions to execute the ENSO processor automatically and verify generated outputs.

### Documentation

This document formalises the processor so that another Data Engineering member can understand the data source, processing logic, outputs, execution procedure and maintenance requirements.

## 19. Reproducibility and Maintainability

The processor is designed around repeatable execution rather than manual data manipulation.

Key reproducibility features include:

- programmatic NOAA data retrieval;
- date-stamped raw files;
- separate processed outputs;
- deterministic feature-generation steps;
- standardised temporal fields;
- documented output schema;
- logging;
- validation checks; and
- GitHub Actions automation.

The raw source preservation also provides an audit trail that can be used to understand which external data was processed for a particular run.

## 20. Assumptions

The current processor documentation assumes:

- NOAA CPC remains the selected ENSO source;
- the documented NOAA endpoint remains available;
- the source format remains compatible with the current parser;
- monthly ENSO observations are represented using month-start alignment;
- `time_id` follows the FireFusion `Time_Registry` convention;
- the processed schema remains compatible with downstream consumers; and
- ENSO classification rules remain aligned with the current implementation.

Any change to the source, schema, classification logic or downstream requirements should be reflected in both the implementation and this documentation.

## 21. Known Limitations

### External Dependency

The pipeline depends on an external NOAA endpoint. Availability or source-format changes may affect execution.

### Monthly Refresh

The source is described as having a monthly refresh frequency, so new information may not be available on every daily automated execution.

### Classification Rules

The current documentation explicitly records the `ONI >= +0.5°C` El Niño condition. The complete threshold definitions for all ENSO phases should be maintained in the implementation and updated here if they change.

### Output Verification

The current GitHub Actions workflow verifies that output directories can be listed. More comprehensive schema and data-quality validation can be added in future iterations.

## 22. Future Improvements

Potential improvements to the ENSO Processor include:

- automated schema validation;
- automated data-quality checks;
- unit tests for date conversion;
- unit tests for lag generation;
- unit tests for ENSO classification;
- stronger HTTP retry and timeout handling;
- validation of duplicate monthly observations;
- automated comparison with the previous successful run;
- automated notifications when the pipeline fails;
- integration testing with the FireFusion fire–climate enrichment workflow; and
- stronger output verification in GitHub Actions.

## 23. Maintenance Checklist

Before merging changes to the ENSO Processor:

- [ ] Confirm the NOAA endpoint is accessible.
- [ ] Inspect any source-format changes.
- [ ] Verify date/season conversion.
- [ ] Verify `datetime_record`.
- [ ] Verify `record_year_month`.
- [ ] Verify `time_id`.
- [ ] Verify `oni_anomaly`.
- [ ] Verify six-month `oni_lag6m`.
- [ ] Verify `enso_phase`.
- [ ] Verify sequential `enso_id`.
- [ ] Verify `original_source`.
- [ ] Check raw file preservation.
- [ ] Check processed CSV generation.
- [ ] Check dataset delivery.
- [ ] Run the processor locally.
- [ ] Run the GitHub Actions workflow.
- [ ] Review generated-file checks.
- [ ] Update this documentation if behaviour changes.

## 24. Evidence for FireFusion Contribution

The following evidence can be attached to the PR or project progress documentation:

| Evidence | What it demonstrates |
|---|---|
| NOAA source inspection | Investigation of the selected external data source |
| `00_inspect_raw_data.py` output | Raw dataset structure and data-quality investigation |
| `01_clean_data.py` output | Repeatable cleaning workflow |
| Processed CSV | Generated ENSO data output |
| `o1_elnino_exploration.ipynb` | Exploratory analysis and understanding of ENSO data |
| Pipeline/log output | Successful processing and observability |
| GitHub Actions run | Automated ENSO execution |
| Generated-file check | Verification of raw/processed/dataset outputs |
| Git branch / PR | Contribution and integration into FireFusion |

> **Recommended PR screenshots:** Include one screenshot of the local processor execution, one of the generated processed CSV/schema, and one successful GitHub Actions run.

## 25. Conclusion

The NOAA CPC ONI ENSO Data Pipeline provides FireFusion with a repeatable mechanism for retrieving and preparing macro-climate information for downstream bushfire-risk analysis.

The workflow covers source extraction, raw-data preservation, temporal alignment, master time-key generation, feature engineering, ENSO phase derivation, primary-key assignment and output delivery.

The development work also established a foundation for automation through GitHub Actions, allowing the processor to be executed on a scheduled basis and checked for expected outputs.

The combination of source lineage, structured processing, documented schema, validation considerations and automation makes the ENSO Processor easier for the Data Engineering stream to reproduce, maintain and integrate with the broader FireFusion pipeline.